In [15]:
from sarigoz.data.fetchers.address_data_fetcher import AddressDataFetcher


In [4]:
import zmq
import json

def get_pools():
    context = zmq.Context()
    socket = context.socket(zmq.REQ)
    socket.setsockopt(zmq.RCVTIMEO, 5000)  # 5 second timeout
    socket.connect("tcp://localhost:5558")
    
    try:
        socket.send_string(json.dumps({'type': 'get_all_pools'}))
        response = json.loads(socket.recv_string())
        
        if response.get('status') == 'success':
            return response.get('data', {})
        return {}
    finally:
        socket.close()
        context.term()

pools = get_pools()
print(pools)

{'0xa7f5BAf528B78D54c1890974Ec228cD6843B639b': {'eth_reserve': 5.982158443266893, 'token_reserve': 12617554.195994861, 'token_address': '0xbF977c4Bd14B59959FfCD9e756646795ae2b2b96', 'pool_address': '0xa7f5BAf528B78D54c1890974Ec228cD6843B639b', 'pool_type': 'V2', 'fee_tier': 3000, 'denom_currency': 'WETH', 'denom_address': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2', 'token_decimals': 18, 'token1_is_denom': True, 'block_number': 22666219, 'update_time': 1749462445.25621, 'token_symbol': 'HOJI', 'token_name': 'Hoji God', 'is_scam': False, 'trading_enabled': False, 'liquidity': 23928.63377306757}, '0xf2D769D8Dd0703413D7f8Ba5fE68F448b833Fc8E': {'eth_reserve': 19.5628921661693, 'token_reserve': 120524200.85422166, 'token_address': '0x182998aCf20ea79FD77995B2BaaC82E056589939', 'pool_address': '0xf2D769D8Dd0703413D7f8Ba5fE68F448b833Fc8E', 'pool_type': 'V2', 'fee_tier': 3000, 'denom_currency': 'WETH', 'denom_address': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2', 'token_decimals': 18, 'token1

In [ ]:

def load_mempool_transactions(jsonl_file):
    """Load transaction hashes from mempool JSONL file"""
    print(f"\n📄 Loading mempool transactions from: {jsonl_file}")
    
    mempool_tx_hashes = set()
    
    try:
        with open(jsonl_file, 'r') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    data = json.loads(line.strip())
                    tx_hash = data.get('tx_hash', '').lower()
                    if tx_hash:
                        mempool_tx_hashes.add(tx_hash)
                except json.JSONDecodeError:
                    continue
                    
                if line_num % 1000 == 0:
                    print(f"  Loaded {line_num} lines...")
        
        print(f"✅ Loaded {len(mempool_tx_hashes)} unique transaction hashes from mempool")
        return mempool_tx_hashes
        
    except Exception as e:
        print(f"❌ Error loading mempool file: {e}")
        return set()

In [5]:
import glob
import os
import re


mempool_pattern = "/home/nima/code/crypto/logs/mempool/mempool_state_changes_*.jsonl"
mempool_files = glob.glob(mempool_pattern)
mempool_files


['/home/nima/code/crypto/logs/mempool/mempool_state_changes_1749493056.jsonl',
 '/home/nima/code/crypto/logs/mempool/mempool_state_changes_1749493283.jsonl',
 '/home/nima/code/crypto/logs/mempool/mempool_state_changes_1749492928.jsonl']

In [8]:
mempool_pattern = "/home/nima/code/crypto/logs/mempool/mempool_state_changes_*.jsonl"
mempool_files = glob.glob(mempool_pattern)
latest_mempool = max(mempool_files, key=os.path.getmtime)
def load_mempool_transactions(jsonl_file):
    """Load transaction hashes from mempool JSONL file"""
    print(f"\n📄 Loading mempool transactions from: {jsonl_file}")
    
    mempool_tx_hashes = {}
    
    try:
        with open(jsonl_file, 'r') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    data = json.loads(line.strip())
                    tx_hash = data.get('tx_hash', '')
                    address = data.get('addresses_with_state_changes', '')
                    if tx_hash:
                        mempool_tx_hashes[tx_hash] = address
                except json.JSONDecodeError:
                    continue
                    
                if line_num % 1000 == 0:
                    print(f"  Loaded {line_num} lines...")
        
        print(f"✅ Loaded {len(mempool_tx_hashes)} unique transaction hashes from mempool")
        return mempool_tx_hashes
        
    except Exception as e:
        print(f"❌ Error loading mempool file: {e}")
        return set()
mempool_txs = load_mempool_transactions(latest_mempool)
mempool_txs


📄 Loading mempool transactions from: /home/nima/code/crypto/logs/mempool/mempool_state_changes_1749493283.jsonl
  Loaded 1000 lines...
  Loaded 2000 lines...
  Loaded 3000 lines...
  Loaded 4000 lines...
  Loaded 5000 lines...
  Loaded 6000 lines...
  Loaded 7000 lines...
  Loaded 8000 lines...
  Loaded 9000 lines...
  Loaded 10000 lines...
  Loaded 11000 lines...
  Loaded 12000 lines...
  Loaded 13000 lines...
  Loaded 14000 lines...
  Loaded 15000 lines...
  Loaded 16000 lines...
  Loaded 17000 lines...
  Loaded 18000 lines...
  Loaded 19000 lines...
  Loaded 20000 lines...
  Loaded 21000 lines...
  Loaded 22000 lines...
  Loaded 23000 lines...
✅ Loaded 23214 unique transaction hashes from mempool


{'...'],
 ...}

In [16]:
from sarigoz.data.fetchers.address_data_fetcher import AddressDataFetcher
adf = AddressDataFetcher()
for pool_addr, pool_info in pools.items():
    pool_txs = adf.get_address_transactions(pool_addr)
    print(pool_txs)
    for tx in pool_txs:
        if tx["tx_hash"] in mempool_txs:
            print(tx["tx_hash"])
    

[{'tx_hash': '0x247e527a862bc772c54fbb467409797a72c340348349566ae42f9f0b0a5e0df9', 'block_number': 22666219, 'from_address': '0x4824F291d29713FC0B33A4b4E0dd735FeD6AbBDa', 'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050', 'value': 0.0, 'status': 'true', 'timestamp': datetime.datetime(2025, 6, 9, 11, 47, 23)}, {'tx_hash': '0x7ccccc6c1cdafdfeb4b871b694468965e00dc98a0837e96279bab57dd9291508', 'block_number': 22666218, 'from_address': '0x510cdccE629d7515e4D4daFD87873E37c66d9a7c', 'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050', 'value': 0.0, 'status': 'true', 'timestamp': datetime.datetime(2025, 6, 9, 11, 47, 11)}, {'tx_hash': '0x49528edd1c254b00fb80275b9645f0f3eeee4103d3291aad5f856735d01aeec5', 'block_number': 22666176, 'from_address': '0xaeda46cEfa1dEC38769E479C87dDB82afCa18b23', 'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050', 'value': 0.0, 'status': 'true', 'timestamp': datetime.datetime(2025, 6, 9, 11, 38, 47)}, {'tx_hash': '0x284ffd665d91ecc89b2293e21ce

In [11]:
pool_txs

[{'tx_hash': '0x247e527a862bc772c54fbb467409797a72c340348349566ae42f9f0b0a5e0df9',
  'block_number': 22666219,
  'from_address': '0x4824F291d29713FC0B33A4b4E0dd735FeD6AbBDa',
  'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050',
  'value': 0.0,
  'status': 'true',
  'timestamp': datetime.datetime(2025, 6, 9, 11, 47, 23)},
 {'tx_hash': '0x7ccccc6c1cdafdfeb4b871b694468965e00dc98a0837e96279bab57dd9291508',
  'block_number': 22666218,
  'from_address': '0x510cdccE629d7515e4D4daFD87873E37c66d9a7c',
  'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050',
  'value': 0.0,
  'status': 'true',
  'timestamp': datetime.datetime(2025, 6, 9, 11, 47, 11)},
 {'tx_hash': '0x49528edd1c254b00fb80275b9645f0f3eeee4103d3291aad5f856735d01aeec5',
  'block_number': 22666176,
  'from_address': '0xaeda46cEfa1dEC38769E479C87dDB82afCa18b23',
  'to_address': '0x8205b4C17c7B4359423fDc2D141140a249B39050',
  'value': 0.0,
  'status': 'true',
  'timestamp': datetime.datetime(2025, 6, 9, 11, 38, 47)},
 {'t

In [ ]:
{"timestamp": "2025-06-09T20:22:36.926900", "tx_hash": "83486cf4520a5c14fd3d75cc1de261236d44f3a2095a7cd741f27253e9a3122a", "addresses_with_state_changes": ["0x7F9379eF44817b288bDeeD560aDB4e16a4ad05CE", "0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49"], "value_wei": "513512458176585500", "gas": "1000000", "input_size": 452}
